In [1]:
import sys
import os
import time
from lightkurve import search_targetpixelfile
from lightkurve import TessTargetPixelFile
import lightkurve as lk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from keras.models import load_model
from keras.optimizers import Adam
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Conv1D, MaxPooling1D, Flatten
from wotan import slide_clip
from wotan import transit_mask, flatten
from astropy.stats import sigma_clip
from astropy import units as u
import csv
import shutil
from scipy.interpolate import interp1d
from tsfresh import extract_features
from tsfresh.utilities.dataframe_functions import make_forecasting_frame
from tsfresh.utilities.dataframe_functions import impute
from statsmodels.tsa.seasonal import seasonal_decompose
from multiprocessing import Pool
import multiprocessing
import numpy as np
import pandas as pd
import lightkurve as lk
from scipy.signal import find_peaks
from astropy.timeseries import BoxLeastSquares
from scipy.interpolate import interp1d

In [2]:
%matplotlib inline

In [3]:
value_df = 10000
cwd = os.getcwd()
dirname = cwd[len(cwd)-len("Satellite Datasets"):len(cwd)]
if(dirname != 'Satellite Datasets'):
    os.chdir('./Satellite Datasets')
star_check = pd.read_csv("exoplanet_star_updated_flux.csv")
star_check = star_check.drop(['tid'],axis=1)
star_check_y = star_check[['confirmed_planet']]
star_check = star_check.reset_index().drop('index',axis=1)

In [5]:
star_check = star_check.drop('confirmed_planet',axis=1).apply(lambda row: row.fillna(0), axis=1)
star_check[['confirmed_planet']] = star_check_y

In [6]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(star_check.drop('confirmed_planet',axis=1),star_check[['confirmed_planet']], test_size=0.1, random_state=42)

In [9]:
from tensorflow.keras.layers import Conv1D, LeakyReLU

model = Sequential([
    Conv1D(filters=16, kernel_size=3, activation='relu', input_shape=(10000,1)),
    MaxPooling1D(pool_size=2),
    Conv1D(filters=32, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Conv1D(filters=64, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [10]:
model.fit(X_train, y_train, epochs=30, batch_size=32, validation_data=(X_test, y_test))

Epoch 1/30
95/95 [==============================] - 6s 65ms/step - loss: 0.7132 - accuracy: 0.5622 - val_loss: 0.6706 - val_accuracy: 0.5697
Epoch 2/30
95/95 [==============================] - 6s 62ms/step - loss: 0.6242 - accuracy: 0.6206 - val_loss: 0.6329 - val_accuracy: 0.6380
Epoch 3/30
95/95 [==============================] - 6s 64ms/step - loss: 0.5718 - accuracy: 0.6975 - val_loss: 0.6257 - val_accuracy: 0.6528
Epoch 4/30
95/95 [==============================] - 6s 63ms/step - loss: 0.5349 - accuracy: 0.7252 - val_loss: 0.6166 - val_accuracy: 0.6499
Epoch 5/30
95/95 [==============================] - 6s 62ms/step - loss: 0.5139 - accuracy: 0.7400 - val_loss: 0.6302 - val_accuracy: 0.6588
Epoch 6/30
95/95 [==============================] - 6s 64ms/step - loss: 0.5045 - accuracy: 0.7479 - val_loss: 0.6275 - val_accuracy: 0.6647
Epoch 7/30
95/95 [==============================] - 6s 63ms/step - loss: 0.4873 - accuracy: 0.7539 - val_loss: 0.6216 - val_accuracy: 0.6647
Epoch 8/30
95

In [ ]:
from tensorflow.keras.models import Model

x_sample = X_test.iloc[1].to_numpy().reshape(1,10000)

layer_outputs = [model.layers[8].output]
activation_model = Model(inputs=model.input, outputs=layer_outputs)

feature_maps = activation_model.predict(x_sample)

first_feature_map = feature_maps[0]
plt.figure(figsize=(20, 15))
for i in range(first_feature_map.shape[-1]):
    plt.subplot(12, 12, i+1)  
    plt.plot(first_feature_map[:, i:i+1])
    plt.axis('off')
plt.show()


In [ ]:
model.layers[21]

In [ ]:
model.evaluate(X_test,y_test)

In [ ]:
model.evaluate(X_test.iloc[2].to_numpy().reshape(1,10000,1),y_test.iloc[2])

In [21]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Dense, Dropout, MaxPooling1D, Flatten, LeakyReLU
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import GridSearchCV

def create_model(conv_config='32', dense_config='64', use_leaky_relu=False, dropout_rate=0.0):
    model = Sequential()
    input_shape = (10000, 1)  # Assuming 10,000 features and 1 channel
    
    # Parse convolutional layers configuration
    conv_filters = [int(x) for x in conv_config.split('-') if x.isdigit()]
    for filters in conv_filters:
        model.add(Conv1D(filters=filters, kernel_size=3, activation='relu', input_shape=input_shape))
        model.add(MaxPooling1D(pool_size=2))
        if use_leaky_relu:
            model.add(LeakyReLU(alpha=0.01))
        if dropout_rate > 0:
            model.add(Dropout(rate=dropout_rate))
        input_shape = None  # After the first layer, no need to specify input_shape
    
    model.add(Flatten())
    
    dense_units = [int(x) for x in dense_config.split('-') if x.isdigit()]
    for units in dense_units:
        model.add(Dense(units=units, activation='relu'))
        if use_leaky_relu:
            model.add(LeakyReLU(alpha=0.01))
        if dropout_rate > 0:
            model.add(Dropout(rate=dropout_rate))
    
    model.add(Dense(1, activation='sigmoid'))  # Assuming binary classification
    
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

conv_layer_configs = ['32', '64', '32-32', '64-64', '32-64-32', '64-32-64', '32-32-32-32', '64-64-64-64', '32-32-32-32-32', '64-64-64-64-64']
dense_layer_configs = ['64', '128', '64-64', '128-128', '64-128-64', '128-64-128', '64-64-64-64', '128-128-128-128']

param_grid = {
    'model__conv_config': conv_layer_configs,
    'model__dense_config': dense_layer_configs,
    'model__use_leaky_relu': [False, True],
    'model__dropout_rate': [0.0, 0.25, 0.5]
}

model = KerasClassifier(model=create_model, epochs=10, verbose=0)

grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3)
# Assume X_train and y_train are defined
grid_result = grid.fit(X_train, y_train)

print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))


2024-03-14 16:21:43.766135: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2024-03-14 16:21:43.766177: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2024-03-14 16:21:43.766425: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2024-03-14 16:21:43.766479: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2024-03-14 16:21:43.766720: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2024-03-14 16:21:43.766765: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2024-03-14 16:21:43.767830: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2024-03-14 16:21:43.767916: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defa

KeyboardInterrupt: 